# 応用編2
このノートブックでは、マルチシグの利用方法を示します。

## 概要
マルチシグは、複数のウォレットでトランザクションに署名を行うことで、セキュリティを高める技術です。  
マルチシグでは、例えば３つのウォレットのうち２つが署名するといった運用も可能です。  
BC+のマルチシグでは、同じユーザに紐づけられた複数のウォレットでトランザクションに署名します。

## 準備

実行の準備を行います。

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { chainID, domain } = await import('../lib/load-config.mjs');
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

## マルチシグ用のウォレットとユーザの作成

マルチシグ用に３つのウォレットを作成します。

In [2]:
var wallet1 = await api.importSigningWallet('es', await api.generateWalletKey('es'));
var wallet2 = await api.importSigningWallet('es', await api.generateWalletKey('es'));
var wallet3 = await api.importSigningWallet('es', await api.generateWalletKey('es'));
var addrs = [ wallet1.address, wallet2.address, wallet3.address ];

マルチシグ用のユーザを作成し、ウォレットを紐づけます。

In [3]:
var resp = await rpc.call(adminWallet, 'c1create', { type: 'user', domain });
var userId = resp.value;
var resp = await rpc.call(adminWallet, 'c1update', { id: userId, prop: 'wallets', value: addrs });
console.log(resp);

{
  txno: 701957,
  txid: 'xXy9HFUwjaEDcgvxqrmEyugBQMeG5njGnPvLBFDmtvfXJB',
  status: 'ok',
  value: null
}


マルチシグの設定をします。  
３つのウォレットのうち、２つのウォレットの署名を必要とする設定にします。

In [4]:
var resp = await rpc.call(adminWallet, 'c1update', { id: userId, prop: 'multisig', value: 2 });
console.log(resp);

{
  txno: 701958,
  txid: 'xkwMFjAPPiW2qjiny5g8YaaViHz9XEjBDjB5BvDUwJDQ2',
  status: 'ok',
  value: null
}


テスト用の簡単なコントラクトをデプロイします。

In [5]:
var cnt = await deploySmartContract(function adv2() {
    return new Date();
});
console.log(cnt);

c036018669


上記作成したユーザによる実行を許可します。

In [6]:
var resp = await rpc.call(adminWallet, 'c1update', { id: cnt, prop: 'accessible_to', value: [userId] });
console.log(resp);

{
  txno: 701962,
  txid: 'x5cB86i37TWndFKjjKyT6WTZt74pCRbeMFxfjepXJq3eRB',
  status: 'ok',
  value: null
}


## トランザクション要求の作成と署名

マルチシグ用のトランザクション要求を作成します。  
３つのウォレットアドレスを配列で指定し、必要な署名の数を２と設定します。

In [7]:
var options = { multisig: 2 };
var request = await api.createRequest(addrs, cnt, {}, options);
console.log(request);

{
  reqbin: Uint8Array(209) [
    123,  34,  99, 111, 110, 116, 114,  97,  99, 116,  34,  58,
     34,  99,  48,  51,  54,  48,  49,  56,  54,  54,  57,  34,
     44,  34,  97, 114, 103, 115,  34,  58, 123, 125,  44,  34,
     97, 100, 100, 114,  34,  58,  91,  34, 101,  67,  87, 104,
     54,  82, 119, 118, 101,  81,  74, 109, 121, 118,  78, 100,
    101, 114,  69,  81,  65, 104, 102,  71,  97,  78,  98,  54,
     66, 102,  34,  44,  34, 101,  57,  80, 118, 102, 110,  72,
    102,  70,  75,  76,  74, 100,  68, 116,  87, 120, 111,  97,
    110,  69,  81, 102,
    ... 109 more items
  ],
  signatures: []
}


トランザクション要求にwallet1で署名します。

In [8]:
await api.signRequest(request, wallet1, chainID);
console.log(request);

{
  reqbin: Uint8Array(209) [
    123,  34,  99, 111, 110, 116, 114,  97,  99, 116,  34,  58,
     34,  99,  48,  51,  54,  48,  49,  56,  54,  54,  57,  34,
     44,  34,  97, 114, 103, 115,  34,  58, 123, 125,  44,  34,
     97, 100, 100, 114,  34,  58,  91,  34, 101,  67,  87, 104,
     54,  82, 119, 118, 101,  81,  74, 109, 121, 118,  78, 100,
    101, 114,  69,  81,  65, 104, 102,  71,  97,  78,  98,  54,
     66, 102,  34,  44,  34, 101,  57,  80, 118, 102, 110,  72,
    102,  70,  75,  76,  74, 100,  68, 116,  87, 120, 111,  97,
    110,  69,  81, 102,
    ... 109 more items
  ],
  signatures: [
    [
      'es',
      Uint8Array(64) [
        161, 221, 164,  66,  61,  61, 231,  54, 224,  57,  42,
        168, 236, 158,  57,  92,  59,   0,   8, 173, 229,   5,
         45, 210, 122, 170, 215,  56,  20, 208,   0, 146, 217,
         26,  89, 128, 194,  42, 228, 108, 169, 246, 240, 192,
         33, 112,  25, 194,  92,  45, 217,  46, 161, 249, 212,
        140,  49, 249,  90,  25,  

トランザクション要求にwallet2で署名します。

In [9]:
await api.signRequest(request, wallet2, chainID);
console.log(request);

{
  reqbin: Uint8Array(209) [
    123,  34,  99, 111, 110, 116, 114,  97,  99, 116,  34,  58,
     34,  99,  48,  51,  54,  48,  49,  56,  54,  54,  57,  34,
     44,  34,  97, 114, 103, 115,  34,  58, 123, 125,  44,  34,
     97, 100, 100, 114,  34,  58,  91,  34, 101,  67,  87, 104,
     54,  82, 119, 118, 101,  81,  74, 109, 121, 118,  78, 100,
    101, 114,  69,  81,  65, 104, 102,  71,  97,  78,  98,  54,
     66, 102,  34,  44,  34, 101,  57,  80, 118, 102, 110,  72,
    102,  70,  75,  76,  74, 100,  68, 116,  87, 120, 111,  97,
    110,  69,  81, 102,
    ... 109 more items
  ],
  signatures: [
    [
      'es',
      Uint8Array(64) [
        161, 221, 164,  66,  61,  61, 231,  54, 224,  57,  42,
        168, 236, 158,  57,  92,  59,   0,   8, 173, 229,   5,
         45, 210, 122, 170, 215,  56,  20, 208,   0, 146, 217,
         26,  89, 128, 194,  42, 228, 108, 169, 246, 240, 192,
         33, 112,  25, 194,  92,  45, 217,  46, 161, 249, 212,
        140,  49, 249,  90,  25,  

## トランザクション要求を送信

ここでは、rpc._call_requestを使って送信する例を示します。

In [10]:
var resp = await rpc._call_request(request);
console.log(resp);

{
  txno: 701963,
  txid: 'x8vTjtM3nFxeWnhV4mv3HxGsy7EF5rFErU32LUG5Duc6y',
  status: 'ok',
  value: '2025-07-25T05:25:27.663Z'
}


## 署名が不足している場合のエラー

In [11]:
request.signatures.length = 1; // signatureを1つにする
var resp = await rpc._call_request(request);
console.log(resp);

RemoteError: insufficient signers

## rpc.callを使った場合のエラー

In [12]:
var resp = await rpc.call(wallet1, cnt, {});
console.log(resp);

{
  txno: 701964,
  txid: 'xhdi6aGDuaqBce4xHq9Coq3ANTyNXCwYa3eEXHSaYi2Uz',
  status: 'denied',
  value: 'insufficient signers for multisig'
}


## クリーンアップ
このノートブックの中で作成したユーザは今後使用しないので、削除しておきます。

In [13]:
var resp = await rpc.call(adminWallet, 'c1terminate', { id: userId });
console.log(resp);

{
  txno: 701965,
  txid: 'xnBAgeURWPSFpMNiWBCRKRB9uz4F9bgxparhiNdiKMfCBB',
  status: 'ok',
  value: 'u48387133'
}
